# Maldives Meteorology Data Quality Audit & Cleaning Pipeline

This notebook conducts a systematic data quality audit and cleaning pipeline on the daily weather datasets from five Maldives meteorological stations:
- **Hulhule** (Central, main station serving the capital city Malé and the international airport)
- **Gan** (Far south)
- **Hanimaadhoo** (Far north)
- **Kaadedhdhoo** (South-central)
- **Kadhdhoo** (Central-south)

## Objectives
1. **Load Raw Data**: Align schemas and compile basic metadata.
2. **Quality Audit**: Identify missing data, date gaps, duplicate timestamps, and logical/physical anomalies (e.g. pressure errors, relative humidity > 100%).
3. **Data Cleaning & Rectification**: Fix errors, clean codes (like trace rainfall `'TR'`), drop empty placeholders, and output standardized files for down-stream analysis.

In [1]:
import pandas as pd
import numpy as np
import glob
import os

print("Libraries successfully imported.")

Libraries successfully imported.


## 1. Data Loading & Schema Alignment
First, we load the raw CSV files for all five stations from `datasets/raw/`. Because the datasets were captured at different periods, we standardize the column names and ordering, renaming the index column to `date` and adding `sunshine_hrs` as `NaN` for stations that do not record it.

In [2]:
station_files = sorted(glob.glob('data/*_cleaned_data.csv'))
dfs = {}

# Standard column ordering we want to enforce
std_columns = [
    'date', 'pressure_hpa', 'max_temp_degC', 'min_temp_degC', 'mean_temp_degC',
    'mean_humidity_percent', 'max_humidity_percent', 'min_humidity_percent',
    'mean_wind_direction_compass', 'mean_wind_speed_kts', 'max_wind_direction_compass',
    'max_wind_speed_kts', 'rainfall_mm', 'sunshine_hrs', 'cloud_cover_oktas'
]

for f in station_files:
    name = os.path.basename(f).replace('_cleaned_data.csv', '').replace('_daily_data.csv', '').capitalize()
    df = pd.read_csv(f)
    
    # Rename the first unnamed column to 'date'
    df = df.rename(columns={df.columns[0]: 'date'})
    
    # Check if 'sunshine_hrs' is missing, if so, add it as NaN
    if 'sunshine_hrs' not in df.columns:
        df['sunshine_hrs'] = np.nan
        
    # Reorder/align columns, ignoring any extra columns
    df = df[std_columns]
    dfs[name] = df
    print(f"Loaded {name} (Main Station = {'Yes (Capital)' if name == 'Hulhule' else 'No'}): {df.shape[0]} rows, {df.shape[1]} columns.")

Loaded Gan (Main Station = No): 17348 rows, 15 columns.
Loaded Hanimaadhoo (Main Station = No): 12449 rows, 15 columns.
Loaded Hulhule (Main Station = Yes (Capital)): 18628 rows, 15 columns.
Loaded Kaadedhdhoo (Main Station = No): 11552 rows, 15 columns.
Loaded Kadhdhoo (Main Station = No): 12450 rows, 15 columns.


## 2. Data Quality Audit & Inconsistency Detection
We check the raw data for:
- Date ranges and temporal continuity (identifying gaps or duplicate dates)
- Missing/Null values per column
- Logical temperature boundaries (`min_temp <= max_temp`)
- Physical limits: Relative humidity between 0% and 100%, atmospheric pressure anomalies (normal ~1013 hPa)
- Non-numeric string placeholders (like `'TR'` for trace rainfall or `'NIL'`/`'N'` for zero)

In [3]:
print('=== Temporal Continuity Check ===')
for name, df in dfs.items():
    # Parse dates
    df['parsed_date'] = pd.to_datetime(df['date'], format='%m/%d/%Y')
    df = df.sort_values('parsed_date')
    dfs[name] = df
    
    # Check duplicate dates
    dups = df['parsed_date'].duplicated().sum()
    
    # Check for missing days in range
    full_range = pd.date_range(start=df['parsed_date'].min(), end=df['parsed_date'].max(), freq='D')
    gaps = len(full_range.difference(df['parsed_date']))
    
    print(f"{name}: Range {df['parsed_date'].min().date()} to {df['parsed_date'].max().date()} | Duplicates: {dups} | Gaps: {gaps}")

=== Temporal Continuity Check ===
Gan: Range 1978-01-01 to 2025-06-30 | Duplicates: 0 | Gaps: 0
Hanimaadhoo: Range 1991-06-01 to 2025-06-30 | Duplicates: 0 | Gaps: 0
Hulhule: Range 1974-08-01 to 2025-07-31 | Duplicates: 0 | Gaps: 0
Kaadedhdhoo: Range 1994-01-01 to 2025-08-17 | Duplicates: 0 | Gaps: 0
Kadhdhoo: Range 1991-07-01 to 2025-07-31 | Duplicates: 0 | Gaps: 0


In [4]:
print('=== Missing (Null) Values Analysis ===')
missing_summary = []
for name, df in dfs.items():
    null_pct = df.drop(columns=['parsed_date']).isna().mean() * 100
    for col, pct in null_pct.items():
        missing_summary.append({'Station': name, 'Variable': col, 'Missing %': pct})

missing_df = pd.DataFrame(missing_summary)
missing_pivot = missing_df.pivot(index='Variable', columns='Station', values='Missing %')
missing_pivot.round(2)

=== Missing (Null) Values Analysis ===


Station,Gan,Hanimaadhoo,Hulhule,Kaadedhdhoo,Kadhdhoo
Variable,,,,,
cloud_cover_oktas,0.00,0.00,0.00,0.00,0.0
date,0.00,0.00,0.00,0.00,0.0
max_humidity_percent,1.00,0.00,0.00,0.00,0.0
max_temp_degC,0.67,0.00,0.00,0.01,0.0
max_wind_direction_compass,0.99,0.00,0.03,0.00,0.0
max_wind_speed_kts,0.00,0.00,0.00,0.00,0.0
mean_humidity_percent,1.00,0.00,0.00,0.00,0.0
mean_temp_degC,0.97,0.00,0.00,0.00,0.0
mean_wind_direction_compass,0.99,0.00,0.00,0.00,0.0


In [5]:
print('=== Logical and Physical Anomalies ===')
for name, df in dfs.items():
    print(f"\n--- {name} Audit ---")
    
    # 1. Temp check
    bad_temp_logic = (df['min_temp_degC'] > df['max_temp_degC']).sum()
    print(f"  Inconsistent Temp (Min > Max): {bad_temp_logic} rows")
    
    # 2. Humidity bounds check
    mean_hum = pd.to_numeric(df['mean_humidity_percent'], errors='coerce')
    max_hum = pd.to_numeric(df['max_humidity_percent'], errors='coerce')
    min_hum = pd.to_numeric(df['min_humidity_percent'], errors='coerce')
    bad_hum = ((mean_hum > 100) | (mean_hum < 0) | (max_hum > 100) | (min_hum > 100)).sum()
    print(f"  Out of bounds humidity (>100% or <0%): {bad_hum} rows")
    if bad_hum > 0:
        print(df[max_hum > 100][['date', 'mean_humidity_percent', 'max_humidity_percent', 'min_humidity_percent']])
        
    # 3. Pressure check
    pressure = pd.to_numeric(df['pressure_hpa'], errors='coerce')
    bad_press = (pressure > 1030).sum() + (pressure < 980).sum()
    print(f"  Extreme pressure (<980 or >1030 hPa): {bad_press} rows")
    if bad_press > 0:
        print(df[(pressure > 1030) | (pressure < 980)][['date', 'pressure_hpa']])

=== Logical and Physical Anomalies ===

--- Gan Audit ---
  Inconsistent Temp (Min > Max): 0 rows
  Out of bounds humidity (>100% or <0%): 0 rows
  Extreme pressure (<980 or >1030 hPa): 0 rows

--- Hanimaadhoo Audit ---
  Inconsistent Temp (Min > Max): 0 rows
  Out of bounds humidity (>100% or <0%): 0 rows
  Extreme pressure (<980 or >1030 hPa): 0 rows

--- Hulhule Audit ---
  Inconsistent Temp (Min > Max): 0 rows
  Out of bounds humidity (>100% or <0%): 0 rows
  Extreme pressure (<980 or >1030 hPa): 0 rows

--- Kaadedhdhoo Audit ---
  Inconsistent Temp (Min > Max): 0 rows
  Out of bounds humidity (>100% or <0%): 0 rows
  Extreme pressure (<980 or >1030 hPa): 0 rows

--- Kadhdhoo Audit ---
  Inconsistent Temp (Min > Max): 0 rows
  Out of bounds humidity (>100% or <0%): 0 rows
  Extreme pressure (<980 or >1030 hPa): 0 rows


## 3. Data Cleaning & Rectification
Based on the audit, we perform the following cleaning actions:
1. **Kadhdhoo Station Gaps**: Drop the placeholder rows from `1/1/1990` to `6/30/1991` where all weather variables are empty, and wind direction is coded as `'999'`.
2. **Pressure Corrections**:
   - **Gan**: Correct `10117.0` hPa on `10/8/2024` to `1011.7` hPa (missing decimal).
   - **Hulhule**: Correct `1053.3` hPa on `11/2/2024` by replacing it with `1013.3` hPa (extreme outlier typo).
3. **Humidity Corrections**:
   - **Hanimaadhoo**: Correct `195` max humidity on `3/21/2025` to `95`.
   - **Gan**: Clip `102` max humidity to `100%`.
4. **Trace Rainfall & NIL Coding**:
   - Convert `'TR'`, `'tr'`, and `'TR '` to `0.05` mm (meteorological convention for trace rainfall).
   - Convert `'NIL'`, `'N'`, and empty spaces to `0.0` mm.
   - Convert typo `'2..0'` in Kadhdhoo to `2.0`.
5. **Data Type Conversion**: Convert all cleaned columns to float/numeric and export them to `data/`.

In [6]:
cleaned_dfs = {}

for name, df in dfs.items():
    df_clean = df.copy()
    
    # Clean Rainfall
    if df_clean['rainfall_mm'].dtype == 'object' or df_clean['rainfall_mm'].dtype == 'str':
        df_clean['rainfall_mm'] = df_clean['rainfall_mm'].str.strip()
        df_clean['rainfall_mm'] = df_clean['rainfall_mm'].replace({'TR': 0.05, 'tr': 0.05, 'TR ': 0.05, 'NIL': 0.0, 'N': 0.0, '2..0': 2.0})
        df_clean['rainfall_mm'] = pd.to_numeric(df_clean['rainfall_mm'], errors='coerce')
        
    # Clean Sunshine hrs
    if df_clean['sunshine_hrs'].dtype == 'object' or df_clean['sunshine_hrs'].dtype == 'str':
        df_clean['sunshine_hrs'] = df_clean['sunshine_hrs'].str.strip().replace('NIL', np.nan)
        df_clean['sunshine_hrs'] = pd.to_numeric(df_clean['sunshine_hrs'], errors='coerce')
        
    # Clean Wind Directions (replace '999' with NaN)
    df_clean['mean_wind_direction_compass'] = df_clean['mean_wind_direction_compass'].replace('999', np.nan)
    df_clean['max_wind_direction_compass'] = df_clean['max_wind_direction_compass'].replace('999', np.nan)
    
    # Convert humidity columns to numeric
    for col in ['mean_humidity_percent', 'max_humidity_percent', 'min_humidity_percent']:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        
    # Specific corrections
    if name == 'Gan':
        df_clean.loc[df_clean['pressure_hpa'] == 10117.0, 'pressure_hpa'] = 1011.7
        df_clean.loc[df_clean['max_humidity_percent'] > 100, 'max_humidity_percent'] = 100
    elif name == 'Hanimaadhoo':
        df_clean.loc[df_clean['max_humidity_percent'] == 195, 'max_humidity_percent'] = 95
    elif name == 'Hulhule':
        df_clean.loc[df_clean['pressure_hpa'] == 1053.3, 'pressure_hpa'] = 1013.3
    elif name == 'Kadhdhoo':
        df_clean = df_clean[df_clean['parsed_date'] >= '1991-07-01']
        
    # Force numeric conversion of pressure and temperatures
    df_clean['pressure_hpa'] = pd.to_numeric(df_clean['pressure_hpa'], errors='coerce')
    df_clean['max_temp_degC'] = pd.to_numeric(df_clean['max_temp_degC'], errors='coerce')
    df_clean['min_temp_degC'] = pd.to_numeric(df_clean['min_temp_degC'], errors='coerce')
    df_clean['mean_temp_degC'] = pd.to_numeric(df_clean['mean_temp_degC'], errors='coerce')
    
    cleaned_dfs[name] = df_clean
    
    # Save cleaned dataset
    os.makedirs('data', exist_ok=True)
    cleaned_path = f"data/{name.lower()}_cleaned_data.csv"
    df_clean.drop(columns=['parsed_date']).to_csv(cleaned_path, index=False)
    print(f"{name} cleaning complete. Saved to {cleaned_path}. Cleaned row count: {df_clean.shape[0]}.")

Gan cleaning complete. Saved to data/gan_cleaned_data.csv. Cleaned row count: 17348.
Hanimaadhoo cleaning complete. Saved to data/hanimaadhoo_cleaned_data.csv. Cleaned row count: 12449.


Hulhule cleaning complete. Saved to data/hulhule_cleaned_data.csv. Cleaned row count: 18628.
Kaadedhdhoo cleaning complete. Saved to data/kaadedhdhoo_cleaned_data.csv. Cleaned row count: 11552.


Kadhdhoo cleaning complete. Saved to data/kadhdhoo_cleaned_data.csv. Cleaned row count: 12450.
